In [1]:
from pathlib import Path

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tardis.io.atom_data import AtomData
from tardis.io.configuration.config_reader import Configuration

home = Path.home()

ION_SLICE = (1, slice(None), slice(None), slice(None))

# identical atomic data to that used by C Vogl
atom_data = AtomData.from_hdf(
    home
    / "tardis-regression-data/atom_data/nlte_atom_data/TestNLTE_He_Ti_ctardis.h5"
)  # currently not available for public use

atom_data.prepare_atom_data([1], 'macroatom', [1], [(1, 0)])

config = Configuration.from_yaml(
    home / "tardis/tardis/plasma/tests/data/plasma_base_test_config.yml"
)

config.supernova.time_explosion = 16.084 * u.day
config.model.structure.type = "file"
config.model.structure.filename = (
    home
    / "tardis/docs/physics/plasma/equilibrium/cmfgen_stephane_density_rebin.dat"
)
config.model.structure.filetype = "simple_ascii"
config.model.structure.v_inner_boundary = 10000 * u.km / u.s
config.model.structure.v_outer_boundary = 15000 * u.km / u.s

config.model.abundances.He = 0
config.model.abundances.H = 1

config.plasma.excitation = "dilute-lte"
config.plasma.ionization = "nebular"

config.plasma.continuum_interaction.species = ["H 1"]
config.plasma.nlte.species = ["H 1"]

/home/afullard/tardis/tardis/__init__.py:17: UserWarning: Astropy is already imported externally. Astropy should be imported after TARDIS.
  warnings.warn(


# Imports

In [2]:
import warnings

from scipy import interpolate
from scipy.optimize import root
from scipy.sparse import coo_matrix

from tardis import constants as const
from tardis.plasma.electron_energy_distribution import (
    ThermalElectronEnergyDistribution,
)
from tardis.plasma.equilibrium.rate_matrix import RateMatrix
from tardis.plasma.equilibrium.rates import (
    CollisionalIonizationSeaton,
    RadiativeRatesSolver,
    SpontaneousRecombinationCoeffSolver,
    ThermalCollisionalRateSolver,
)


# Indexing

In [3]:
def calculate_lower_ion_level_index(
    level_number_density: pd.DataFrame,
):
    """Calculate index for lower ion levels (ion_number == 0).

    Parameters
    ----------
    lte_level_number_density : pd.DataFrame
        DataFrame containing level number densities with a MultiIndex
        that includes 'ion_number'.

    Returns
    -------
    pd.Series
        Boolean Series indicating rows corresponding to lower ion levels.
    """
    return level_number_density.index.get_level_values("ion_number") == 0

def calculate_upper_ion_population_index(
    ion_number_density: pd.DataFrame,
):
    """Calculate index for upper ion populations (ion_number > 0).

    Parameters
    ----------
    lte_ion_number_density : pd.DataFrame
        DataFrame containing ion number densities with a MultiIndex
        that includes 'ion_number'.

    Returns
    -------
    pd.Series
        Boolean Series indicating rows corresponding to upper ion populations.
    """
    return ion_number_density.index.get_level_values("ion_number") > 0

Identical

In [4]:
def calculate_block_ids_from_dataframe(dataframe):
    block_start_id = (
        np.where(np.diff(dataframe.index.get_level_values(0)) != 0.0)[0] + 1
    )
    return np.hstack(([0], block_start_id, [len(dataframe)]))

In [5]:
def calculate_lines_lower_level_index(levels, lines):
    levels_index = pd.Series(
        np.arange(len(levels), dtype=np.int64), index=levels
    )
    lines_index = lines.index.droplevel("level_number_upper")
    return np.array(levels_index.loc[lines_index])

def calculate_lines_upper_level_index(levels, lines):
    levels_index = pd.Series(
        np.arange(len(levels), dtype=np.int64), index=levels
    )
    lines_index = lines.index.droplevel("level_number_lower")
    return np.array(levels_index.loc[lines_index])

# Simple properties

Identical

In [6]:
def calculate_beta_temperature(temperature):
    return 1 / (const.k_B.cgs.value * temperature)

In [7]:
def calculate_g_electron(beta_temperature):
    return (
        (2 * np.pi * const.m_e.cgs.value / beta_temperature)
        / (const.h.cgs.value**2)
    ) ** 1.5

In [8]:
def calculate_partition_function(level_boltzmann_factor):
    return level_boltzmann_factor.groupby(
        level=["atomic_number", "ion_number"]
    ).sum()

# Ion number density

Identical to `IonNumberDensity.calculate_with_n_electron`

In [9]:
def calculate_ion_number_density(
    saha_factor,
    partition_function,
    elemental_number_density,
    electron_number_density,
    block_ids,
    ion_zero_threshold,
):
    if block_ids is None:
        block_ids = calculate_block_ids_from_dataframe(saha_factor)

    ion_populations = np.empty_like(partition_function.values)

    phi_electron = np.nan_to_num(saha_factor.values / electron_number_density.values)

    for i, start_id in enumerate(block_ids[:-1]):
        end_id = block_ids[i + 1]
        current_phis = phi_electron[start_id:end_id]
        phis_product = np.cumprod(current_phis, 0)

        tmp_ion_populations = np.empty(
            (current_phis.shape[0] + 1, current_phis.shape[1])
        )
        tmp_ion_populations[0] = elemental_number_density.values[i] / (
            1 + np.sum(phis_product, axis=0)
        )
        tmp_ion_populations[1:] = tmp_ion_populations[0] * phis_product

        ion_populations[start_id + i : end_id + 1 + i] = tmp_ion_populations

    ion_populations[ion_populations < ion_zero_threshold] = 0.0

    return pd.DataFrame(data=ion_populations, index=partition_function.index)

# Level number density

Identical to dilute LTE case

In [10]:
def initialize_indices(levels, partition_function):
    indexer = pd.Series(
        np.arange(partition_function.shape[0]),
        index=partition_function.index,
    )
    return indexer.loc[levels.droplevel(2)].values


def calculate_level_number_density(
    level_boltzmann_factor,
    ion_number_density,
    levels,
    partition_function,
):
    """
    Calculate the level populations from the level_boltzmann_factor,
    ion_number_density and partition_function
    """
    ion2level_idx = initialize_indices(levels, partition_function)

    partition_function_broadcast = partition_function.values[ion2level_idx]
    level_population_fraction = (
        level_boltzmann_factor.values / partition_function_broadcast
    )
    ion_number_density_broadcast = ion_number_density.values[ion2level_idx]
    level_number_density = (
        level_population_fraction * ion_number_density_broadcast
    )
    return pd.DataFrame(
        level_number_density, index=level_boltzmann_factor.index
    )

# LTE properties

Identical except for a `.fillna(0.0)` on the output

In [11]:
def calculate_saha_factor_lte(
    g_electron, beta_radiation, partition_function, ionization_data
):
    saha_factors = np.empty(
        (
            partition_function.shape[0]
            - partition_function.index.get_level_values(0).unique().size,
            partition_function.shape[1],
        )
    )

    block_ids = calculate_block_ids_from_dataframe(partition_function)

    for i, start_id in enumerate(block_ids[:-1]):
        end_id = block_ids[i + 1]
        current_block = partition_function.values[start_id:end_id]
        current_saha_factors = current_block[1:] / current_block[:-1]
        saha_factors[start_id - i : end_id - i - 1] = current_saha_factors

    broadcast_ionization_energy = ionization_data.reindex(
        partition_function.index
    ).dropna()
    saha_factor_index = broadcast_ionization_energy.index
    broadcast_ionization_energy = broadcast_ionization_energy.values

    saha_factor_coefficient = (
        2
        * g_electron
        * np.exp(np.outer(broadcast_ionization_energy, -beta_radiation))
    )

    return pd.DataFrame(saha_factors * saha_factor_coefficient, index=saha_factor_index)

Identical

In [12]:
def calculate_lte_level_boltzmann_factor(excitation_energy, g, beta_radiation, levels):
    exponential = np.exp(np.outer(excitation_energy.values, -beta_radiation))
    level_boltzmann_factor_array = g.values[np.newaxis].T * exponential
    level_boltzmann_factor = pd.DataFrame(
        level_boltzmann_factor_array,
        index=levels,
        columns=np.arange(len(beta_radiation)),
        dtype=np.float64,
    )
    return level_boltzmann_factor

Identical

In [13]:
def calculate_dilute_lte_level_boltzmann_factor(excitation_energy, g, beta_radiation, levels, dilution_factor, metastability):
    level_boltzmann_factor = calculate_lte_level_boltzmann_factor(excitation_energy, g, beta_radiation, levels)
    level_boltzmann_factor[~metastability] *= dilution_factor
    return level_boltzmann_factor

Identical

In [14]:
def calculate_lte_properties(
    levels,
    electron_phi_lte,
    electron_lte_partition_function,
    electron_lte_level_boltzmann_factor,
    elemental_number_density,
    electron_number_density: pd.Series,
):
    # defined as calculate_with_n_electron at the saha factor in LTE at electron temperature,
    # and at the LTE partition function at electron temperature,
    lte_ion_number_density = calculate_ion_number_density(
        electron_phi_lte,
        electron_lte_partition_function,
        elemental_number_density,
        electron_number_density,
        None,
        1e-20, # default value from tardis
    )

    # defined as calculate_dilute_lte at the LTE level boltzmann factor at electron temperature,
    # the LTE ion number density,
    # and the LTE partition function at electron temperature
    lte_level_number_density = calculate_level_number_density(
        electron_lte_level_boltzmann_factor,
        lte_ion_number_density,
        levels,
        electron_lte_partition_function,
    )
    return lte_ion_number_density, lte_level_number_density

# Stimulated emission factor

Identical

In [15]:
def get_g_lower(g, lines_lower_level_index):
        g_lower = np.array(
            g.iloc[lines_lower_level_index], dtype=np.float64
        )
        _g_lower = g_lower[np.newaxis].T
        return _g_lower

def get_g_upper(g, lines_upper_level_index):
    g_upper = np.array(
        g.iloc[lines_upper_level_index], dtype=np.float64
    )
    _g_upper = g_upper[np.newaxis].T
    return _g_upper

def get_metastable_upper(metastability, lines_upper_level_index):
    _meta_stable_upper = metastability.values[
        lines_upper_level_index
    ][np.newaxis].T
    return _meta_stable_upper

def calculate_stimulated_emission_factor(
        g,
        level_number_density,
        lines_lower_level_index,
        lines_upper_level_index,
        metastability,
    ):
    n_lower = level_number_density.values.take(
        lines_lower_level_index, axis=0, mode="raise"
    )
    n_upper = level_number_density.values.take(
        lines_upper_level_index, axis=0, mode="raise"
    )
    g_lower = get_g_lower(g, lines_lower_level_index)
    g_upper = get_g_upper(g, lines_upper_level_index)
    meta_stable_upper = get_metastable_upper(
        metastability, lines_upper_level_index
    )

    # In theory the factor should be 1 for n_lower = 0, but in practice the opacity is reduced to 0 anyway
    stimulated_emission_factor = np.zeros(n_lower.shape, dtype=np.float64)

    n_lower_zero_mask = n_lower == 0.0
    stimulated_emission_factor[~n_lower_zero_mask] = 1 - (
        (g_lower * n_upper)[~n_lower_zero_mask]
        / (g_upper * n_lower)[~n_lower_zero_mask]
    )

    # the following line probably can be removed as well
    stimulated_emission_factor[np.isneginf(stimulated_emission_factor)] = (
        0.0
    )
    stimulated_emission_factor[
        meta_stable_upper & (stimulated_emission_factor < 0)
    ] = 0.0
    return stimulated_emission_factor

# Phi Saha (nebular approximation)

Identical

In [16]:
def _set_chi_0(chi_0_species, ionization_data):
    if chi_0_species == (20, 2):
        chi_0 = 1.9020591570241798e-11
    else:
        chi_0 = ionization_data.loc[chi_0_species]
    return chi_0


def calculate_radiation_field_correction(
        dilution_factor, ionization_data, beta_rad, electron_temperature, radiation_temperature, beta_electron, chi_0_species
    ):
    chi_0 = _set_chi_0(chi_0_species, ionization_data)
    departure_coefficient = (
        1.0 / dilution_factor
    )  # see Equation 13 and explanations on page 451 lower right in ML 93
    radiation_field_correction = -np.ones(
        (len(ionization_data), len(beta_rad))
    )
    less_than_chi_0 = (ionization_data < chi_0).values
    factor_a = electron_temperature / (departure_coefficient * dilution_factor * radiation_temperature)
    radiation_field_correction[~less_than_chi_0] = factor_a * np.exp(
        np.outer(
            ionization_data.values[~less_than_chi_0],
            beta_rad - beta_electron,
        )
    )
    radiation_field_correction[less_than_chi_0] = 1 - np.exp(
        np.outer(ionization_data.values[less_than_chi_0], beta_rad)
        - beta_rad * chi_0
    )
    radiation_field_correction[less_than_chi_0] += factor_a * np.exp(
        np.outer(ionization_data.values[less_than_chi_0], beta_rad)
        - chi_0 * beta_electron
    )

    radiation_correction_df = pd.DataFrame(
        radiation_field_correction,
        columns=np.arange(len(radiation_temperature)),
        index=ionization_data.index,
    )
    return radiation_correction_df

Identical

In [17]:
def get_zeta_values(zeta_data, ion_index, t_rad):
    zeta_t_rad = zeta_data.columns.values.astype(np.float64)
    zeta_values = zeta_data.loc[ion_index].values.astype(np.float64)
    zeta = interpolate.interp1d(
        zeta_t_rad, zeta_values, bounds_error=False, fill_value=np.nan
    )(t_rad)
    zeta = zeta.astype(float)

    if np.any(np.isnan(zeta)):
        warnings.warn(
            f"t_rads outside of zeta factor interpolation"
            f" zeta_min={zeta_data.columns.values.min():.2f} zeta_max={zeta_data.columns.values.max():.2f} "
            f"- replacing with zeta = 1.0"
        )
        zeta[np.isnan(zeta)] = 1.0

    return zeta

Identical after testing

In [18]:
def calculate_phi_saha_nebular(
    radiative_temperature,
    dilution_factor,
    zeta_data,
    electron_temperature,
    radiation_field_correction,
    g_electron,
    beta_rad,
    partition_function,
    ionization_data,
):
    phi_lte = calculate_saha_factor_lte(
        g_electron, beta_rad, partition_function, ionization_data
    )
    zeta = get_zeta_values(zeta_data, phi_lte.index, radiative_temperature)
    phis = (
        phi_lte
        * dilution_factor
        * ((zeta * radiation_field_correction) + dilution_factor * (1 - zeta))
        * (electron_temperature / radiative_temperature) ** 0.5
    )
    return phis

In [19]:
t_rad = t_electrons = 9992.27229695
w = 0.3571996
delta = 1.0
g_electron = g_electron_Te = 2.41188339e+21
beta_rad = beta_electron = 7.24857452e+11
partition_function = pd.read_csv("/home/afullard/tardis-chvogl-configs/test_values_NLTE_iteration/partition_function.csv", index_col=(0, 1))
lte_partition_function_Te = lte_partition_function = pd.read_csv("/home/afullard/tardis-chvogl-configs/test_values_NLTE_iteration/lte_partition_function_Te.csv", index_col=(0, 1))
partition_function.columns = partition_function.columns.astype(int)
lte_partition_function_Te.columns = lte_partition_function_Te.columns.astype(int)
lte_partition_function.columns = lte_partition_function.columns.astype(int)

phi_nebular = 1.542107e+14

In [20]:
calculate_phi_saha_nebular(t_rad, w, atom_data.zeta_data, t_electrons, delta, g_electron, beta_rad, partition_function.loc[:, [0]], atom_data.ionization_data)

,,0
atomic_number,ion_number,
1,1,1.542106e+14


phi_lte = 6.679977e+14

In [21]:
calculate_saha_factor_lte(g_electron, beta_rad, partition_function.loc[:, [0]], atom_data.ionization_data)

,,0
atomic_number,ion_number,
1,1,6.679976e+14


phi_Te = 3.335294e+14

In [22]:
calculate_saha_factor_lte(g_electron_Te, beta_electron, lte_partition_function_Te.loc[:, [0]], atom_data.ionization_data)

,,0
atomic_number,ion_number,
1,1,3.335294e+14


# Probably need to break this up in some way

Look for repetition

Make a Sobolev-scaled rate matrix solver version

# NLTE Boltzmann factor

In [23]:
number_density = 2.206919e+09
species = (1, 0)
phis = 15375.64125101

Identical but needs unit testing

In [24]:
def solve_ionization_factor(species, saha_factor_per_electron, number_density, saha_factor_nlte):
    atomic_number, ion_number = species

    ionization_factor_matrix = np.diag(np.ones(atomic_number), k=1)
    ionization_factor_matrix[-1] = 1.0  # Number conservation constraint

    diag_indices = np.diag_indices(atomic_number)
    ionization_factor_matrix[diag_indices] = -saha_factor_per_electron
    ionization_factor_matrix[ion_number, ion_number] = -saha_factor_nlte

    number_conservation = np.zeros(len(ionization_factor_matrix))
    number_conservation[-1] = number_density

    return np.linalg.solve(ionization_factor_matrix, number_conservation)

ion_numbers = array([1.43524101e+05, 2.20677509e+09])

In [25]:
solve_ionization_factor(species, phis, number_density, phis)

array([1.43524126e+05, 2.20677548e+09])

Similar but really needs unit testing

In [26]:
from tardis.opacities.tau_sobolev import (
    calculate_beta_sobolev,
    calculate_sobolev_line_opacity,
)


def solve_boltzmann_factor(
    trial_value,
    species,
    phi_saha_per_electron,
    elemental_number_density,
    lte_level_number_density,
    lte_ion_number_density,
    thermal_electron_distribution,
    g,
    lines_lower_level_index,
    lines_upper_level_index,
    lower_ion_level_index,
    upper_ion_population_index,
    metastability,
    lines,
    time_explosion,
    number_of_levels,
    radiative_rate_matrix_solver,
    estimated_radiation_field,
    electron_distribution,
    collisional_excitation_rate_matrix_df,
    collisional_ionization_rate,
    previous_electron_densities,
    estimated_photoionization_rate,
    estimated_stim_recomb_rate,
    spontaneous_recomb_rate,
):
    ion_number = solve_ionization_factor(
        species,
        phi_saha_per_electron,
        elemental_number_density.loc[species[0], 0],
        trial_value[-1],
    )

    level_number_density = ion_number[species[1]] * trial_value[:-1] # scale level number density by lower ion level population

    level_number_density = pd.DataFrame(
        level_number_density,
        index=lte_level_number_density.loc[
            (species[0], species[1], slice(None)), :
        ].index,
    )

    stimulated_emission_factor = calculate_stimulated_emission_factor(
        g,
        level_number_density,
        lines_lower_level_index,
        lines_upper_level_index,
        metastability,
    )

    tau_sobolevs = calculate_sobolev_line_opacity(
        lines,
        level_number_density,
        time_explosion,
        stimulated_emission_factor,
    )

    beta_sobolevs = calculate_beta_sobolev(tau_sobolevs)

    beta_sobolev_matrix_ul = coo_matrix(
        (
            beta_sobolevs[0],
            (
                beta_sobolevs.index.get_level_values("level_number_upper"),
                beta_sobolevs.index.get_level_values("level_number_lower"),
            ),
        ),
        shape=(number_of_levels, number_of_levels),
    )

    beta_sobolev_matrix_lu = coo_matrix(
        (
            beta_sobolevs[0],
            (
                beta_sobolevs.index.get_level_values("level_number_lower"),
                beta_sobolevs.index.get_level_values("level_number_upper"),
            ),
        ),
        shape=(number_of_levels, number_of_levels),
    )

    radiative_excitation_rate_matrix_df = radiative_rate_matrix_solver.solve(
        estimated_radiation_field, electron_distribution
    )

    radiative_excitation_rate_matrix = radiative_excitation_rate_matrix_df.loc[
        species, 0
    ]

    # one shell only. Lucy 2003 eq 10
    radiative_excitation_rate_matrix *= (
        beta_sobolev_matrix_ul + beta_sobolev_matrix_lu
    ).toarray()

    excitation_rate_matrix = (
        radiative_excitation_rate_matrix
        + collisional_excitation_rate_matrix_df.loc[species, 0]
    )

    np.fill_diagonal(
        excitation_rate_matrix, -np.sum(excitation_rate_matrix, axis=0)
    )

    excitation_rate_matrix[0, :] = 1.0

    level_to_ion_population_factor = pd.DataFrame(
        lte_level_number_density.loc[lower_ion_level_index].values
        / (
            lte_ion_number_density.loc[upper_ion_population_index].values
            * thermal_electron_distribution.number_density.value
        ),
        index=lte_level_number_density.loc[lower_ion_level_index].index,
    )

    collisional_recomb_rate = (
        collisional_ionization_rate
        * level_to_ion_population_factor
        * previous_electron_densities**2
    )

    ionization_rate_vector = (
        estimated_photoionization_rate + collisional_ionization_rate
    )
    ionization_rate_vector.iloc[0] = 0

    ionization_rate_matrix = -np.diag(ionization_rate_vector[0].values)

    recombination_rate_vector = (
        estimated_stim_recomb_rate
        + spontaneous_recomb_rate
        + collisional_recomb_rate
    )
    total_inverse_recombination_rate = -recombination_rate_vector.sum()

    total_rate_matrix = np.append(
        excitation_rate_matrix + ionization_rate_matrix,
        np.expand_dims(ionization_rate_vector[0].values, 1),
        axis=1,
    )

    total_rate_matrix = np.append(
        total_rate_matrix,
        [
            np.hstack(
                [
                    ionization_rate_vector[0].values,
                    total_inverse_recombination_rate,
                ]
            )
        ],
        axis=0,
    )

    number_conservation_vec = np.zeros(total_rate_matrix.shape[0])
    number_conservation_vec[0] = 1.0

    matrix_solution = (
        np.dot(total_rate_matrix, trial_value) - number_conservation_vec
    )
    # solutions = np.linalg.solve(total_rate_matrix, number_conservation_vec)
    return matrix_solution

Approximately the same?

In [27]:
def calculate_level_boltzmann_factor(
    atomic_data,
    lines_lower_level_index,
    lines_upper_level_index,
    lower_ion_level_index,
    upper_ion_population_index,
    g,
    metastability,
    lines,
    species,
    electron_distribution,
    estimated_radiation_field,
    previous_ion_number_density,
    previous_electron_densities,
    rad_field_mc_estimators,
    lte_ion_number_density,
    lte_level_number_density,
    previous_level_number_density,
    elemental_number_density,
    phi_saha_nebular,
    time_explosion,
    radiative_rate_matrix_solver,
    collisional_rate_matrix_solver,
    collisional_ionization_coeff_solver,
    spontaneous_recomb_rate_solver
):
    species_slice = (species[0], species[1], slice(None), slice(None))

    phi_saha_per_electron = (
        phi_saha_nebular.loc[species].values
        / previous_electron_densities.values
    )

    collisional_ionization_rate = collisional_ionization_coeff_solver.solve(
        electron_distribution.temperature
    )

    estimated_photoionization_rate = rad_field_mc_estimators.photo_ion_estimator
    estimated_stim_recomb_rate = (
        rad_field_mc_estimators.stim_recomb_estimator
        * previous_electron_densities
    )

    spontaneous_recomb_rate = (
        spontaneous_recomb_rate_solver.solve(electron_distribution.temperature)
        * previous_electron_densities
    )

    collisional_excitation_rate_matrix_df = (
        collisional_rate_matrix_solver.solve(
            estimated_radiation_field, electron_distribution
        )
    )

    number_of_levels = atomic_data.levels.energy.loc[species].count()

    next_ion_index = (species[0], species[1] + 1)

    previous_ion_population = (
        previous_ion_number_density.loc[next_ion_index, 0]
        / previous_ion_number_density.loc[species, 0]
    )

    initial_guess = previous_level_number_density[0].loc[species].values
    initial_guess /= initial_guess.sum()
    initial_guess = np.hstack([initial_guess, previous_ion_population])

    args = (
        species,
        phi_saha_per_electron,
        elemental_number_density,
        lte_level_number_density,
        lte_ion_number_density,
        electron_distribution,
        g,
        lines_lower_level_index,
        lines_upper_level_index,
        lower_ion_level_index,
        upper_ion_population_index,
        metastability,
        lines,
        time_explosion,
        number_of_levels,
        radiative_rate_matrix_solver,
        estimated_radiation_field,
        electron_distribution,
        collisional_excitation_rate_matrix_df,
        collisional_ionization_rate,
        previous_electron_densities,
        estimated_photoionization_rate,
        estimated_stim_recomb_rate,
        spontaneous_recomb_rate,
    )

    solutions = root(solve_boltzmann_factor, x0=initial_guess, args=args)

    level_boltzmann_factor = solutions.x[:-1]
    #ion_ratio[i] = solutions.x[-1]

    general_level_boltzmann_factor = pd.DataFrame(level_boltzmann_factor,
                index=atomic_data.levels.loc[species_slice, :].index,
                columns=np.arange(len(previous_electron_densities)),
                dtype=np.float64)

    general_level_boltzmann_factor.loc[(1, 1, 0), 0] = 1.0 #hack

    return general_level_boltzmann_factor

In [28]:
class PreviousIterationProperties:
    def __init__(
        self,
        ion_number_density,
        level_number_density,
        electron_number_density,
        level_boltzmann_factor,
        partition_function
    ):
        self.ion_number_density = ion_number_density
        self.level_number_density = level_number_density
        self.electron_number_density = electron_number_density
        self.partition_function = partition_function
        self.level_boltzmann_factor = level_boltzmann_factor

In [29]:
def calculate_electron_density_fractional_heating(
    inputs,
    species,
    atomic_data,
    lines_lower_level_index,
    lines_upper_level_index,
    radiation_field,
    estimated_radiation_field,
    ion_solver,
    thermal_solver,
    max_electron_density,
    ion_slice,
    elemental_number_density,
    time_simulation,
    time_explosion,
    volume,
    previous_iteration_properties,
    radiative_rate_matrix_solver,
    collisional_rate_matrix_solver,
    collisional_ionization_rate_coeff_solver,
    spontaneous_recomb_rate_solver,
    collisional_bound_rate_solver,
    ff_heating_estimator,
    bf_heating_estimator,
    stim_recomb_cooling_estimator,
    rad_field_mc_estimators
):
    fractional_electron_density = inputs[::2]
    link_t_rad_t_electron = inputs[1::2]

    previous_electron_densities = (
        fractional_electron_density * max_electron_density.values
    )
    electron_temperature = radiation_field.temperature * link_t_rad_t_electron

    thermal_electron_distribution = ThermalElectronEnergyDistribution(
        0 * u.erg,
        electron_temperature,
        previous_electron_densities * u.cm**-3,
    )

    levels = atomic_data.levels.loc[ion_slice, :].index
    lines = atomic_data.lines.loc[ion_slice, :]
    excitation_energy = atomic_data.levels.loc[ion_slice, :]["energy"]
    metastability = atomic_data.levels.loc[ion_slice, :]["metastable"]
    g = atomic_data.levels.loc[ion_slice, :]["g"]

    ionization_data = atomic_data.ionization_data

    beta_rad_temperature = calculate_beta_temperature(radiation_field.temperature.value)
    beta_electron_temperature = calculate_beta_temperature(electron_temperature.value)

    # electron temperature-based calculations
    g_electron = calculate_g_electron(beta_rad_temperature)
    electron_lte_level_boltzmann_factor = calculate_lte_level_boltzmann_factor(excitation_energy, g, beta_rad_temperature, levels)
    electron_lte_partition_function = calculate_partition_function(electron_lte_level_boltzmann_factor)
    electron_phi_lte = calculate_saha_factor_lte(g_electron, beta_electron_temperature, electron_lte_partition_function, ionization_data)

    lte_ion_number_density, lte_level_number_density = (
        calculate_lte_properties(
            levels,
            electron_phi_lte,
            electron_lte_partition_function,
            electron_lte_level_boltzmann_factor,
            elemental_number_density,
            pd.Series(previous_electron_densities),)
    )

    ion_number_density, electron_number_density = ion_solver.solve_estimated(
        thermal_electron_distribution,
        rad_field_mc_estimators,
        elemental_number_density,
        time_simulation,
        volume,
        lte_level_number_density,
        previous_iteration_properties.level_number_density,
        lte_ion_number_density,
        previous_iteration_properties.ion_number_density,
        previous_iteration_properties.partition_function,
        previous_iteration_properties.level_boltzmann_factor,
        tolerance=1e-8,
    )

    radiation_field_correction = calculate_radiation_field_correction(
        radiation_field.dilution_factor,
        ionization_data,
        beta_rad_temperature,
        electron_temperature,
        radiation_field.temperature,
        beta_electron_temperature,
        (1, 1), # Hydrogen for Type II
    )

    phi_saha_nebular = calculate_phi_saha_nebular(
        radiation_field.temperature,
        radiation_field.dilution_factor,
        atomic_data.zeta_data,
        electron_temperature,
        radiation_field_correction,
        g_electron,
        beta_rad_temperature,
        electron_lte_partition_function,
        ionization_data,
    )

    lower_ion_level_index = calculate_lower_ion_level_index(
        lte_level_number_density
    )
    upper_ion_population_index = calculate_upper_ion_population_index(
        lte_ion_number_density
    )

    level_boltzmann_factor = calculate_level_boltzmann_factor(
        atomic_data, # basically the same
        lines_lower_level_index, # should be the same
        lines_upper_level_index, # should be the same
        lower_ion_level_index, # seems correct
        upper_ion_population_index, # seems correct
        g, # matches
        metastability, # matches
        lines, # same except for size
        species, # same
        thermal_electron_distribution, # identical
        estimated_radiation_field, # should be the same
        previous_iteration_properties.ion_number_density, # not identical
        electron_number_density, # not quite the same after going through the ion solver
        rad_field_mc_estimators, # identical
        lte_ion_number_density, # not matching
        lte_level_number_density, # not matching
        previous_iteration_properties.level_number_density, # should be normalized? not matching
        elemental_number_density, # identical
        phi_saha_nebular, # ~2x too large
        time_explosion, # identical
        radiative_rate_matrix_solver,
        collisional_rate_matrix_solver,
        collisional_ionization_rate_coeff_solver,
        spontaneous_recomb_rate_solver)

    partition_function = calculate_partition_function(level_boltzmann_factor)
    level_number_density = calculate_level_number_density(level_boltzmann_factor, ion_number_density, levels, partition_function)

    previous_iteration_properties.level_number_density = level_number_density
    previous_iteration_properties.ion_number_density = ion_number_density
    previous_iteration_properties.electron_number_density = electron_number_density
    previous_iteration_properties.level_boltzmann_factor = level_boltzmann_factor
    previous_iteration_properties.partition_function = partition_function

    fractional_electron_density_change = (
        electron_number_density
        - thermal_electron_distribution.number_density.value
    ) / thermal_electron_distribution.number_density.value

    thermal_electron_distribution.number_density = (
        electron_number_density.to_numpy() * u.cm**-3
    )

    collisional_ionization_rate_coeff = collisional_ionization_rate_coeff_solver.solve(thermal_electron_distribution.temperature)
    collisional_bound_rate_coeff = collisional_bound_rate_solver.solve(
        thermal_electron_distribution.temperature
    )

    level_to_ion_population_factor = pd.DataFrame(
        lte_level_number_density.loc[lower_ion_level_index].values
        / (
            lte_ion_number_density.loc[upper_ion_population_index].values
            * thermal_electron_distribution.number_density.value
        ),
        index=lte_level_number_density.loc[lower_ion_level_index].index,
    )

    heating_rate, fractional_heating_rate = thermal_solver.solve(
        thermal_electron_distribution,
        level_number_density.loc[:, [0]],
        ion_number_density.loc[:, [0]],
        collisional_ionization_rate_coeff.loc[:, [0]],
        collisional_bound_rate_coeff.iloc[419:, [0]],
        collisional_bound_rate_coeff.iloc[:419, [0]],
        ff_heating_estimator[0],
        level_to_ion_population_factor.loc[:, [0]],
        bound_free_heating_estimator=bf_heating_estimator.loc[:, [0]],
        stimulated_recombination_estimator=stim_recomb_cooling_estimator.loc[
            :, [0]
        ],
    )

    print(
        "Fractional change in e- density:\n",
        fractional_electron_density_change.values,
    )
    print("Fractional heating rate:\n", fractional_heating_rate.values)
    print("Temperature:\n", electron_temperature)
    output = np.zeros(2 * len(radiation_field.temperature))
    output[::2] = fractional_electron_density_change.values
    output[1::2] = fractional_heating_rate.values
    return output

# Setup

In [30]:
from tardis.plasma.radiation_field import (
    DilutePlanckianRadiationField,
)

species = (1, 0)  # Hydrogen

lines_lower_level_index = calculate_lines_lower_level_index(
    atom_data.levels.index,
    atom_data.lines
)

lines_upper_level_index = calculate_lines_upper_level_index(
    atom_data.levels.index,
    atom_data.lines
)

radiation_temp = 9992.27229695 * np.ones(1) * u.K
dilution_factor = 0.3571996 * np.ones(1)

electron_temp = radiation_temp.value
electron_density = 2206775091.3630457 * np.ones(1)

elemental_number_density = pd.DataFrame(2.20691862e+09 * np.ones(1), index=[1])
elemental_number_density.index.name = "atomic_number"
radiation_field = DilutePlanckianRadiationField(radiation_temp, dilution_factor)

max_electron_density = (elemental_number_density * elemental_number_density.index.values).sum()
fractional_electron_density = electron_density / max_electron_density

time_simulation = 7.2671371e-44 * u.s
volume = 1.61751052e44 * np.ones(1) * u.cm**3

In [31]:
ctardis_lines = pd.read_csv("/home/afullard/tardis-chvogl-configs/ctardis_lines.csv", index_col=(0,))

In [32]:
# Find lines present in ctardis_lines but not in plasma atom_data lines
plasma_lines = atom_data.lines.loc[ION_SLICE, :]

# Create MultiIndex for ctardis_lines to match plasma lines structure
ctardis_multiindex = pd.MultiIndex.from_arrays([
    ctardis_lines['atomic_number'].values,
    ctardis_lines['ion_number'].values,
    ctardis_lines['level_number_lower'].values.astype(int),
    ctardis_lines['level_number_upper'].values.astype(int)
], names=['atomic_number', 'ion_number', 'level_number_lower', 'level_number_upper'])

ctardis_lines_indexed = ctardis_lines.copy()
ctardis_lines_indexed.index = ctardis_multiindex

# Find lines in ctardis but not in plasma
ctardis_only = ctardis_lines_indexed.index.difference(plasma_lines.index)
print(f"Lines in ctardis_lines but not in plasma.atom_data.lines: {len(ctardis_only)}")

# Find lines present in both ctardis and plasma
common_lines = ctardis_lines_indexed.index.intersection(plasma_lines.index)
print(f"Lines in both ctardis_lines and plasma.atom_data.lines: {len(common_lines)}")

# Create list of indices for ctardis_lines that correspond to lines present in atom_data.lines
common_indices = []
for idx, line_index in enumerate(ctardis_lines_indexed.index):
    if line_index in plasma_lines.index:
        common_indices.append(idx)

print(f"Number of common line indices: {len(common_indices)}")

Lines in ctardis_lines but not in plasma.atom_data.lines: 16
Lines in both ctardis_lines and plasma.atom_data.lines: 419
Number of common line indices: 419


In [33]:
radiative_transitions = atom_data.lines.loc[ION_SLICE, :]

radiative_rate_solver = RadiativeRatesSolver(radiative_transitions)

col_strength_temperatures = atom_data.collision_data_temperatures

if atom_data.collision_data == "dummy value":
    col_strengths = atom_data.yg_data.loc[atom_data.lines.loc[ION_SLICE, :].index] # handles the issue that there is more collision than line data
    col_type = "cmfgen"
else:
    col_strengths = atom_data.collision_data.loc[ION_SLICE, :]
    col_type = "chianti"


collisional_rate_solver = ThermalCollisionalRateSolver(
    atom_data.levels,
    radiative_transitions,
    col_strength_temperatures,
    col_strengths,
    col_type,
)

radiative_rate_solvers = [
    (radiative_rate_solver, "radiative"),
]

collisional_rate_solvers = [
    (collisional_rate_solver, "electron"),
]

radiative_rate_matrix_solver = RateMatrix(
    radiative_rate_solvers, atom_data.levels
)
collisional_rate_matrix_solver = RateMatrix(
    collisional_rate_solvers, atom_data.levels
)

collisional_ionization_rate_coeff_solver = CollisionalIonizationSeaton(
    atom_data.photoionization_data
)

spontaneous_recomb_rate_solver = SpontaneousRecombinationCoeffSolver(
        atom_data.photoionization_data
    )

# Estimators

In [34]:
class DummyEstimators:
    def __init__(self, photo_ion_estimator, stim_recomb_estimator):
        self.photo_ion_estimator = photo_ion_estimator
        self.stim_recomb_estimator = stim_recomb_estimator

photo_ion_estimator = pd.read_csv("/home/afullard/tardis-chvogl-configs/photo_ion_estimator.csv", index_col=(0))
stim_recomb_estimator = pd.read_csv("/home/afullard/tardis-chvogl-configs/stim_recomb_estimator.csv", index_col=(0))

photo_ion_estimator.columns = photo_ion_estimator.columns.astype(int)
stim_recomb_estimator.columns = stim_recomb_estimator.columns.astype(int)
# Create MultiIndex for photo_ion_estimator
photo_ion_estimator_idx = pd.MultiIndex.from_tuples(
    [(1, 0, level) for level in photo_ion_estimator.index],
    names=['atomic_number', 'ion_number', 'level_number']
)
photo_ion_estimator.index = photo_ion_estimator_idx

# Create MultiIndex for stim_recomb_estimator
stim_recomb_estimator_idx = pd.MultiIndex.from_tuples(
    [(1, 0, level) for level in stim_recomb_estimator.index],
    names=['atomic_number', 'ion_number', 'level_number']
)
stim_recomb_estimator.index = stim_recomb_estimator_idx

rad_field_mc_estimators = DummyEstimators(photo_ion_estimator.loc[:, [0]], stim_recomb_estimator.loc[:, [0]])

class EstimatedRadiationField:
    def __init__(self, temperature, j_blues):
        self.j_blues = j_blues
        self.temperature = temperature

    def calculate_mean_intensity(self, nu):
        return self.j_blues.values

j_blues_ctardis = pd.read_csv("/home/afullard/tardis-chvogl-configs/j_blues.csv", index_col=0)
estimated_radiation_field = EstimatedRadiationField(9992.27229695 * np.ones(1) * u.K, j_blues_ctardis.iloc[common_indices, 0])

In [35]:
data_path = home / "tardis-regression-data/testdata/thermal_data"
bf_heating_estimator = pd.read_csv(data_path / "thermal_bf_heating_est.csv", index_col=(0, 1, 2))
stim_recomb_cooling_estimator = pd.read_csv(data_path / "thermal_stim_cooling_est.csv", index_col=(0, 1, 2))
level_population_ratio = pd.read_csv(data_path / "thermal_level_pop_ratio.csv", index_col=(0, 1, 2))
coll_exc_coeff = pd.read_csv(data_path / "thermal_coll_exc_coeff.csv", index_col=(0, 1, 2, 3))
coll_deexc_coeff = pd.read_csv(data_path / "thermal_coll_deexc_coeff.csv", index_col=(0, 1, 2, 3))
coll_ion_rate_coeff = pd.read_csv(data_path / "thermal_coll_ion_rate_coeff.csv", index_col=(0, 1, 2))

ff_heating_estimator = [  4.89135279e-24,   4.37696370e-24,   3.75869301e-24,
         4.97847160e-24,   4.52158002e-24,   4.21024499e-24,
         3.94991540e-24,   3.72915649e-24,   3.58902110e-24,
         3.40170224e-24,   3.20848519e-24,   3.03540032e-24,
         2.87314722e-24,   2.74328938e-24,   2.61063140e-24,
         2.50640248e-24,   2.38164559e-24,   2.26967531e-24,
         2.24509826e-24,   2.12378192e-24,   2.02063266e-24,
         1.92509873e-24,   1.83070678e-24,   1.77346374e-24]

# because pandas reads in the columns as strings, we need to convert them back to integers
bf_heating_estimator.columns = bf_heating_estimator.columns.astype(int)
stim_recomb_cooling_estimator.columns = stim_recomb_cooling_estimator.columns.astype(int)
level_population_ratio.columns = level_population_ratio.columns.astype(int)
coll_exc_coeff.columns = coll_exc_coeff.columns.astype(int)
coll_deexc_coeff.columns = coll_deexc_coeff.columns.astype(int)
coll_ion_rate_coeff.columns = coll_ion_rate_coeff.columns.astype(int)

# Solve

In [36]:
from tardis.plasma.equilibrium.ion_populations import IonPopulationSolver
from tardis.plasma.equilibrium.rate_matrix import IonRateMatrix
from tardis.plasma.equilibrium.rates import (
    AnalyticPhotoionizationRateSolver,
    CollisionalIonizationRateSolver,
    EstimatedPhotoionizationRateSolver,
)
from tardis.plasma.equilibrium.rates.heating_cooling_rates import (
    BoundFreeThermalRates,
    CollisionalBoundThermalRates,
    CollisionalIonizationThermalRates,
    FreeFreeThermalRates,
)
from tardis.plasma.equilibrium.thermal_balance import ThermalBalanceSolver

analytic_photoionization_rate_solver = AnalyticPhotoionizationRateSolver(atom_data.photoionization_data)
estimated_photoionization_rate_solver = EstimatedPhotoionizationRateSolver(atom_data.photoionization_data, atom_data.level2continuum_edge_idx)

collisional_ionization_rate_solver = CollisionalIonizationRateSolver(atom_data.photoionization_data)

estimated_ion_rate_matrix_solver = IonRateMatrix(estimated_photoionization_rate_solver, collisional_ionization_rate_solver)
estimated_ion_number_density_solver = IonPopulationSolver(estimated_ion_rate_matrix_solver)

bf_rates = BoundFreeThermalRates(atom_data.photoionization_data)
ff_rates = FreeFreeThermalRates()
coll_ion_rates = CollisionalIonizationThermalRates(atom_data.photoionization_data)
coll_bound_rates = CollisionalBoundThermalRates(atom_data.lines.loc[ION_SLICE, :])

thermal_solver = ThermalBalanceSolver(bf_rates, ff_rates, coll_ion_rates, coll_bound_rates)

beta_electron = calculate_beta_temperature(electron_temp)

electron_lte_level_boltzmann_factor = calculate_lte_level_boltzmann_factor(
    atom_data.levels.loc[ION_SLICE, :]["energy"],
    atom_data.levels.loc[ION_SLICE, :]["g"],
    beta_electron,
    atom_data.levels.loc[ION_SLICE, :].index,
)

electron_lte_partition_function = calculate_partition_function(electron_lte_level_boltzmann_factor)

electron_phi_lte = calculate_saha_factor_lte(
    calculate_g_electron(beta_electron),
    beta_electron,
    electron_lte_partition_function,
    atom_data.ionization_data
)


lte_estimated_ion_number_density, lte_estimated_level_number_density = calculate_lte_properties(
    atom_data.levels.loc[ION_SLICE, :].index,
    electron_phi_lte,
    electron_lte_partition_function,
    electron_lte_level_boltzmann_factor,
    elemental_number_density,
    pd.Series(electron_density),
)

lte_partition_function = calculate_partition_function(electron_lte_level_boltzmann_factor)

previous_iteration_properties = PreviousIterationProperties(lte_estimated_ion_number_density, lte_estimated_level_number_density, electron_density, electron_lte_level_boltzmann_factor, lte_partition_function)

# Level number density is solved at least once with a 1.0 link_t_rad_t_electron

In [ ]:
levels = atom_data.levels.loc[ION_SLICE, :].index
lines = atom_data.lines.loc[ION_SLICE, :]
excitation_energy = atom_data.levels.loc[ION_SLICE, :]["energy"]
metastability = atom_data.levels.loc[ION_SLICE, :]["metastable"]
g = atom_data.levels.loc[ION_SLICE, :]["g"]

ionization_data = atom_data.ionization_data

beta_rad_temperature = calculate_beta_temperature(
    radiation_field.temperature.value
)
beta_electron_temperature = calculate_beta_temperature(electron_temp)

# electron temperature-based calculations
g_electron = calculate_g_electron(beta_rad_temperature)
electron_lte_level_boltzmann_factor = calculate_lte_level_boltzmann_factor(
    excitation_energy, g, beta_rad_temperature, levels
)
electron_lte_partition_function = calculate_partition_function(
    electron_lte_level_boltzmann_factor
)
electron_phi_lte = calculate_saha_factor_lte(
    g_electron,
    beta_electron_temperature,
    electron_lte_partition_function,
    ionization_data,
)

thermal_electron_distribution = ThermalElectronEnergyDistribution(
    0 * u.erg,
    electron_temp * u.K,
    electron_density * u.cm**-3,
)

lte_ion_number_density, lte_level_number_density = calculate_lte_properties(
    levels,
    electron_phi_lte,
    electron_lte_partition_function,
    electron_lte_level_boltzmann_factor,
    elemental_number_density,
    pd.Series(electron_density),
)

ion_number_density, electron_number_density = (
    estimated_ion_number_density_solver.solve_estimated(
        thermal_electron_distribution,
        rad_field_mc_estimators,
        elemental_number_density,
        time_simulation,
        volume,
        lte_level_number_density,
        previous_iteration_properties.level_number_density,
        lte_ion_number_density,
        previous_iteration_properties.ion_number_density,
        previous_iteration_properties.partition_function,
        previous_iteration_properties.level_boltzmann_factor,
        tolerance=1e-8,
    )
)

radiation_field_correction = calculate_radiation_field_correction(
    radiation_field.dilution_factor,
    ionization_data,
    beta_rad_temperature,
    electron_temp,
    radiation_field.temperature.value,
    beta_electron_temperature,
    (1, 1),  # Hydrogen for Type II
)

phi_saha_nebular = calculate_phi_saha_nebular(
    radiation_field.temperature,
    radiation_field.dilution_factor,
    atom_data.zeta_data,
    electron_temp,
    radiation_field_correction,
    g_electron,
    beta_rad_temperature,
    electron_lte_partition_function,
    ionization_data,
)

lower_ion_level_index = calculate_lower_ion_level_index(
    lte_level_number_density
)
upper_ion_population_index = calculate_upper_ion_population_index(
    lte_ion_number_density
)

level_boltzmann_factor = calculate_level_boltzmann_factor(
    atom_data,
    lines_lower_level_index,
    lines_upper_level_index,
    lower_ion_level_index,
    upper_ion_population_index,
    g,
    metastability,
    lines,
    species,
    thermal_electron_distribution,
    estimated_radiation_field,
    lte_ion_number_density,  # "previous" ion number density
    electron_number_density,
    rad_field_mc_estimators,
    lte_ion_number_density,
    lte_level_number_density,
    lte_level_number_density,  # "previous" level number density, maybe? There's a weird loop between the NLTE level boltzmann factor and level number density
    elemental_number_density,
    phi_saha_nebular,
    config.supernova.time_explosion,
    radiative_rate_matrix_solver,
    collisional_rate_matrix_solver,
    collisional_ionization_rate_coeff_solver,
    spontaneous_recomb_rate_solver,
)

In [ ]:
level_boltzmann_factor

In [ ]:
from scipy.optimize import least_squares
from scipy.sparse import block_diag

initial = np.zeros(2 * len(radiation_field.dilution_factor))
initial[::2] = fractional_electron_density.values
initial[1::2] = radiation_field.dilution_factor ** 0.25

jac_sparsity = block_diag([np.ones((2, 2))] * 1)

# Temperature to aim for: 8174K

In [ ]:
result = least_squares(calculate_electron_density_fractional_heating,
              initial,
              bounds=([0.0, 0.15], [1.0, 1.0]),
              args=(species,
                    atom_data,
                    lines_lower_level_index,
                    lines_upper_level_index,
                    radiation_field,
                    estimated_radiation_field,
                    estimated_ion_number_density_solver,
                    thermal_solver,
                    max_electron_density,
                    ION_SLICE,
                    elemental_number_density,
                    time_simulation,
                    config.supernova.time_explosion,
                    volume,
                    previous_iteration_properties,
                    radiative_rate_matrix_solver,
                    collisional_rate_matrix_solver,
                    collisional_ionization_rate_coeff_solver,
                    spontaneous_recomb_rate_solver,
                    collisional_rate_solver,
                    ff_heating_estimator,
                    bf_heating_estimator,
                    stim_recomb_cooling_estimator,
                    rad_field_mc_estimators),
              xtol=1e-14,
              ftol=1e-12,
              gtol=1e-14,
              x_scale='jac',
              jac_sparsity=jac_sparsity,
              verbose=2,
              max_nfev=100)